# Ejercicio 11: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [2]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"
model = SentenceTransformer(MODEL_NAME, device="cuda")

C:\Users\FREDDY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4221.16it/s]


In [3]:
from bs4 import BeautifulSoup

file = 'c:\\Users\\FREDDY\\Documents\\GitHub\\RECUPERACIO_DE_LA_INFORMACION\\11Web_Scrapping\\data\\rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [4]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [5]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [6]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [11]:
import os
from bs4 import BeautifulSoup
import json

# Ruta donde tienes los 17 archivos HTML
CARPETA_HTML = r"c:\Users\FREDDY\Documents\GitHub\RECUPERACIO_DE_LA_INFORMACION\11Web_Scrapping\data"

corpus = []  # Aquí se guardarán todas las recetas

print(f"Buscando archivos HTML en: {CARPETA_HTML}")

# Recorremos todos los archivos .html de la carpeta
for archivo in os.listdir(CARPETA_HTML):
    if archivo.endswith(".html"):
        ruta_completa = os.path.join(CARPETA_HTML, archivo)
        print(f"Procesando: {archivo}")

        try:
            with open(ruta_completa, "r", encoding="utf-8") as f:
                html_content = f.read()

            soup = BeautifulSoup(html_content, "html.parser")

            # 1. Título (meta og:title)
            title_tag = soup.find("meta", {"property": "og:title"})
            titulo = title_tag["content"] if title_tag else archivo.replace(".html", "")

            # 2. Descripción (meta name="description")
            desc_tag = soup.find("meta", {"name": "description"})
            descripcion = desc_tag["content"] if desc_tag else ""

            # 3. Ingredientes
            ingredientes_items = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
            ingredientes = [i.get_text().strip() for i in ingredientes_items]

            # 4. Instrucciones (pueden variar, probamos dos selectores)
            instrucciones_items = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
            if not instrucciones_items:
                ol = soup.find("ol", class_="comp mntl-sc-block-group--OL")
                if ol:
                    instrucciones_items = ol.find_all("li")
            instrucciones = [i.get_text().strip() for i in instrucciones_items]

            # 5. Nutrición (opcional)
            nutri_items = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
            nutricion = {}
            for item in nutri_items:
                parent = item.parent
                if parent:
                    texto = parent.get_text().strip().replace('\n', ' ')
                    partes = texto.split(' ', 1)
                    if len(partes) == 2:
                        nutricion[partes[0]] = partes[1]

            # Guardamos la receta en el corpus
            corpus.append({
                "url": f"local/{archivo}",
                "title": titulo,
                "description": descripcion,
                "ingredients": ingredientes,
                "instructions": instrucciones,
                "nutrition": nutricion
            })

            print(f"{titulo} - {len(ingredientes)} ingredientes, {len(instrucciones)} pasos")

        except Exception as e:
            print(f"Error procesando {archivo}: {e}")

print(f"\nCorpus construido con {len(corpus)} recetas.")

# Guardamos el corpus en un JSON para reutilizarlo
with open("corpus_recetas.json", "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)
print("Corpus guardado en 'corpus_recetas.json'")

Buscando archivos HTML en: c:\Users\FREDDY\Documents\GitHub\RECUPERACIO_DE_LA_INFORMACION\11Web_Scrapping\data
Procesando: Air-Fryer-BBQ Baby-Back-Ribs-Recipe.html
Air Fryer BBQ Baby Back Ribs - 11 ingredientes, 5 pasos
Procesando: Baked-BBQ-Chicken-Drumsticks-Recipe.html
Baked BBQ Chicken Drumsticks - 9 ingredientes, 5 pasos
Procesando: Basic-Air-Fryer-Hot-Dogs-Recipe.html
Basic Air Fryer Hot Dogs - 2 ingredientes, 8 pasos
Procesando: BBQ-Chicken-Breasts-in-the-Oven-Recipe.html
BBQ Chicken Breasts in the Oven - 3 ingredientes, 7 pasos
Procesando: Carolina-Style_Whole-Hog_Barbecue-Pork-Recipe.html
Carolina-Style "Whole Hog" Barbecue Pork - 14 ingredientes, 13 pasos
Procesando: Easy-Oven-Barbecued-Beef-Brisket-Recipe.html
Easy Oven-Barbecued Beef Brisket - 4 ingredientes, 3 pasos
Procesando: Instant-Pot-Crispy-Barbecue-Chicken-Wings-Recipe.html
Instant Pot Crispy Barbecue Chicken Wings - 4 ingredientes, 7 pasos
Procesando: Instant-Pot-Pulled-Pork-Sandwiches-Recipe.html
Instant Pot Pulle

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google import genai  

#### 1. CONFIGURACIÓN DE GEMINI


In [ ]:
API_KEY = os.environ.get("GEMINI_API_KEY")
cliente_gemini = genai.Client(api_key=API_KEY)

#### 2. CREAR EMBEDDINGS DEL CORPUS

In [28]:
print("Generando embeddings de las recetas...")
modelo_emb = SentenceTransformer('all-MiniLM-L6-v2')

textos_corpus = []
for receta in corpus:
    titulo = receta.get('title', '')
    desc = receta.get('description', '')
    ingredientes = ' '.join(receta.get('ingredients', []))
    instrucciones = ' '.join(receta.get('instructions', []))
    texto_completo = f"{titulo}. {desc}. Ingredientes: {ingredientes}. Instrucciones: {instrucciones}"
    textos_corpus.append(texto_completo)

embeddings_corpus = modelo_emb.encode(textos_corpus, show_progress_bar=True)
print(f"{len(embeddings_corpus)} embeddings generados.")

Generando embeddings de las recetas...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.92it/s]

17 embeddings generados.


#### 3. FUNCIÓN DE BÚSQUEDA (RETRIEVAL)

In [29]:
def buscar_recetas(pregunta, top_k=3):
    emb_pregunta = modelo_emb.encode([pregunta])
    similitudes = cosine_similarity(emb_pregunta, embeddings_corpus)[0]
    indices_top = np.argsort(similitudes)[::-1][:top_k]
    resultados = []
    for idx in indices_top:
        receta = corpus[idx]
        resultados.append({
            'puntuacion': float(similitudes[idx]),
            'titulo': receta['title'],
            'descripcion': receta['description'],
            'ingredientes': receta['ingredients'],
            'instrucciones': receta['instructions'][:3]  # Solo primeras 3
        })
    return resultados

#### 4. FUNCIÓN DE GENERACIÓN AUMENTADA (RAG + GEMINI)

In [36]:
def responder_con_gemini(pregunta, top_k=2):
    recetas_relevantes = buscar_recetas(pregunta, top_k=top_k)
    if not recetas_relevantes:
        return "No encontré recetas relacionadas con tu consulta."

    contexto = "A continuación tienes información de recetas similares a tu consulta:\n\n"
    for i, r in enumerate(recetas_relevantes):
        contexto += f"--- Receta {i+1}: {r['titulo']} (Relevancia: {r['puntuacion']:.2f}) ---\n"
        contexto += f"Descripción: {r['descripcion']}\n"
        contexto += f"Ingredientes: {', '.join(r['ingredientes'][:6])}\n"
        contexto += f"Instrucciones resumidas: {'. '.join(r['instrucciones'])}\n\n"

    prompt = f"""
    {contexto}
    Basándote EXCLUSIVAMENTE en la información de las recetas proporcionadas arriba, responde a la siguiente pregunta de forma clara, útil y concisa.
    Si la pregunta pide una receta, recomienda la más adecuada y explica por qué.
    Si no puedes responder con la información dada, dilo honestamente.

    Pregunta del usuario: {pregunta}

    Respuesta:
    """

    try:
        respuesta = cliente_gemini.models.generate_content(
            model="gemini-2.5-flash",  # O "gemini-1.5-flash"
            contents=prompt
        )
        return respuesta.text
    except Exception as e:
        return f"Error al llamar a Gemini: {e}"

#### 5. EJEMPLO DE USO

In [37]:
pregunta = "¿Cómo hacer un pollo jugoso y crujiente?"
print(f"\nPregunta: {pregunta}")
print("\n--- TOP 3 RECETAS RECUPERADAS ---")
resultados = buscar_recetas(pregunta, top_k=3)
for i, r in enumerate(resultados):
    print(f"\n{i+1}. {r['titulo']} (Score: {r['puntuacion']:.4f})")
    print(f"   Ingredientes: {', '.join(r['ingredientes'][:3])}...")

print("\n" + "-"*60)
print("RESPUESTA GENERADA POR GEMINI:")
print("-"*60)
respuesta_final = responder_con_gemini(pregunta, top_k=2)
print(respuesta_final)


Pregunta: ¿Cómo hacer un pollo jugoso y crujiente?

--- TOP 3 RECETAS RECUPERADAS ---

1. Zesty Slow Cooker Chicken Barbecue (Score: 0.0551)
   Ingredientes: 6  frozen skinless, boneless chicken breast halves, 1 (12 ounce) bottle barbeque sauce, ½ cup Italian salad dressing...

2. Slow Cooker Barbecue Chicken Breast (Score: 0.0097)
   Ingredientes: 2  boneless chicken breasts, ½ teaspoon salt, ½ teaspoon ground black pepper...

3. Instant Pot Crispy Barbecue Chicken Wings (Score: -0.0163)
   Ingredientes: 1 cup water, 2 ½ pounds frozen chicken wings, 1 cup barbeque sauce...

------------------------------------------------------------
RESPUESTA GENERADA POR GEMINI:
------------------------------------------------------------
Basándome EXCLUSIVAMENTE en la información de las recetas proporcionadas, no puedo darte instrucciones sobre cómo hacer pollo "crujiente". Ambas recetas son para pollo en olla de cocción lenta (slow cooker/crockpot), un método de cocción que típicamente produce po